# 02 Features
Pre-race features only. Every history feature uses `shift(1)`, so a race never sees its own result.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW_DATA_DIR / "races_2022_2025.csv", parse_dates=["EventDate"])
raw = raw.sort_values(["EventDate", "GridPosition"]).reset_index(drop=True)
raw.shape

(1836, 14)

## Team names
Same team, new name across seasons. Map to one ID so team form carries over.

In [2]:
TEAM_MAP = {
    "Red Bull Racing": "Red Bull",
    "Alpine F1 Team": "Alpine",
    "AlphaTauri": "RB",
    "RB F1 Team": "RB",
    "Alfa Romeo": "Sauber",
}
raw["Team"] = raw["TeamName"].replace(TEAM_MAP)
raw["Team"].value_counts()

Team
Ferrari         184
Red Bull        184
Mercedes        184
Sauber          184
Alpine          184
RB              184
McLaren         184
Haas F1 Team    183
Williams        183
Aston Martin    182
Name: count, dtype: int64

## Features
- **Known before the race:** grid, pit-lane start, gap to pole, teammate grid gap
- **From past races only:** driver form, team form, races so far

In [3]:
FORM_WINDOW = 5

def add_quali_features(df):
    df = df.copy()
    pole_time = df.groupby(["Year", "Round"])["BestQualiTime"].transform("min")
    df["GapToPolePct"] = (df["BestQualiTime"] / pole_time - 1) * 100
    # No time set -> slowest in that race; whole race missing (Miami 2025) -> NaN for now
    race_max = df.groupby(["Year", "Round"])["GapToPolePct"].transform("max")
    df["GapToPolePct"] = df["GapToPolePct"].fillna(race_max)
    df["NoQualiTime"] = df["BestQualiTime"].isna().astype(int)

    teammate_grid = df.groupby(["Year", "Round", "Team"])["GridPosition"].transform("sum") - df["GridPosition"]
    df["GridVsTeammate"] = df["GridPosition"] - teammate_grid
    return df


def past_rolling(series, window=FORM_WINDOW):
    # shift(1): current race excluded
    return series.shift(1).rolling(window, min_periods=1).mean()


def add_driver_form(df):
    df = df.sort_values("EventDate").copy()
    by_driver = df.groupby("Abbreviation")
    df["DriverPodiumRate"] = by_driver["Podium"].transform(past_rolling)
    df["DriverAvgFinish"] = by_driver["FinishingPosition"].transform(past_rolling)
    df["DriverAvgGrid"] = by_driver["GridPosition"].transform(past_rolling)
    df["RacesSoFar"] = by_driver.cumcount()
    return df


def add_team_form(df):
    team_race = (
        df.groupby(["Team", "EventDate"], as_index=False)
        .agg(TeamPodiums=("Podium", "sum"), TeamAvgFinishRace=("FinishingPosition", "mean"))
        .sort_values("EventDate")
    )
    by_team = team_race.groupby("Team")
    team_race["TeamPodiumRate"] = by_team["TeamPodiums"].transform(past_rolling)
    team_race["TeamAvgFinish"] = by_team["TeamAvgFinishRace"].transform(past_rolling)
    return df.merge(team_race[["Team", "EventDate", "TeamPodiumRate", "TeamAvgFinish"]],
                    on=["Team", "EventDate"], how="left", validate="many_to_one")


def build_features(raw):
    df = add_quali_features(raw)
    df = add_driver_form(df)
    df = add_team_form(df)
    return df.sort_values(["EventDate", "GridPosition"]).reset_index(drop=True)

features = build_features(raw)
features.head()

,Year,Round,GrandPrix,EventDate,Abbreviation,FullName,TeamName,GridPosition,FinishingPosition,Status,...,Team,GapToPolePct,NoQualiTime,GridVsTeammate,DriverPodiumRate,DriverAvgFinish,DriverAvgGrid,RacesSoFar,TeamPodiumRate,TeamAvgFinish
0,2022,1,Bahrain Grand Prix,2022-03-20,LEC,Charles Leclerc,Ferrari,1.0,1.0,Finished,...,Ferrari,0.000000,0,-2.0,NaN,NaN,NaN,0,NaN,NaN
1,2022,1,Bahrain Grand Prix,2022-03-20,VER,Max Verstappen,Red Bull Racing,2.0,19.0,Fuel pressure,...,Red Bull,0.135825,0,-2.0,NaN,NaN,NaN,0,NaN,NaN
2,2022,1,Bahrain Grand Prix,2022-03-20,SAI,Carlos Sainz,Ferrari,3.0,2.0,Finished,...,Ferrari,0.142450,0,2.0,NaN,NaN,NaN,0,NaN,NaN
3,2022,1,Bahrain Grand Prix,2022-03-20,PER,Sergio Perez,Red Bull Racing,4.0,18.0,Fuel pressure,...,Red Bull,0.400848,0,2.0,NaN,NaN,NaN,0,NaN,NaN
4,2022,1,Bahrain Grand Prix,2022-03-20,HAM,Lewis Hamilton,Mercedes,5.0,3.0,Finished,...,Mercedes,0.541090,0,-4.0,NaN,NaN,NaN,0,NaN,NaN


In [4]:
FEATURES = [
    "GridPosition", "PitLaneStart", "GapToPolePct", "NoQualiTime", "GridVsTeammate",
    "DriverPodiumRate", "DriverAvgFinish", "DriverAvgGrid", "RacesSoFar",
    "TeamPodiumRate", "TeamAvgFinish",
]
features[FEATURES].isna().sum()

GridPosition         0
PitLaneStart         0
GapToPolePct        20
NoQualiTime          0
GridVsTeammate       0
DriverPodiumRate    31
DriverAvgFinish     31
DriverAvgGrid       31
RacesSoFar           0
TeamPodiumRate      20
TeamAvgFinish       20
dtype: int64

NaNs in form features = driver/team's first race (no history). Models handle that in notebook 04.

## Leakage test
Change one race's results. Its own features must not move; only later races may change.

In [5]:
target = features.query("Year == 2023 and Round == 10")
key = ["Year", "Round", "Abbreviation"]

tampered = raw.copy()
mask = (tampered["Year"] == 2023) & (tampered["Round"] == 10)
tampered.loc[mask, "FinishingPosition"] = tampered.loc[mask, "FinishingPosition"].values[::-1]
tampered.loc[mask, "Podium"] = (tampered.loc[mask, "FinishingPosition"] <= 3).astype(int)
features_tampered = build_features(tampered)

before = target.set_index(key)[FEATURES].sort_index()
after = features_tampered.query("Year == 2023 and Round == 10").set_index(key)[FEATURES].sort_index()
pd.testing.assert_frame_equal(before, after)

next_race_changed = not features.query("Year == 2023 and Round == 11")[FEATURES].equals(
    features_tampered.query("Year == 2023 and Round == 11")[FEATURES])
print("Same-race features unchanged: OK")
print("Next race features changed:", next_race_changed)

Same-race features unchanged: OK
Next race features changed: True


## Sanity check
Podium rate should rise as grid improves and recent form gets better.

In [6]:
print(features.groupby("GridPosition")["Podium"].mean().head(10).round(2))
print(features[FEATURES + ["Podium"]].corr()["Podium"].sort_values().round(2))

GridPosition
1.0     0.80
2.0     0.74
3.0     0.48
4.0     0.37
5.0     0.22
6.0     0.11
7.0     0.08
8.0     0.01
9.0     0.02
10.0    0.03
Name: Podium, dtype: float64
GridPosition       -0.52
DriverAvgGrid      -0.51
DriverAvgFinish    -0.51
TeamAvgFinish      -0.50
GapToPolePct       -0.23
GridVsTeammate     -0.20
PitLaneStart       -0.04
NoQualiTime        -0.03
RacesSoFar          0.08
TeamPodiumRate      0.54
DriverPodiumRate    0.54
Podium              1.00
Name: Podium, dtype: float64


In [7]:
KEEP = ["Year", "Round", "GrandPrix", "EventDate", "Abbreviation", "Team"] + FEATURES + ["FinishingPosition", "Podium"]
features[KEEP].to_csv(PROCESSED_DATA_DIR / "features.csv", index=False)
features[KEEP].shape

(1836, 19)